# Convert EHR data to format used in this repo

In [ ]:
# connect to EMAP
# retrieve bed moves
# identify moment of exit from ED/SDEC
# set up snapshot datetimes to sample
# sample ED visits at those times

## DB connection script

In [16]:
# Set up database connection
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

def get_credentials():
    with open('../../secret-emap', 'r') as file:
        username = file.readline().strip()
        password = file.readline().strip()
        database_host = file.readline().strip()
        database_name = file.readline().strip()
        database_port = file.readline().strip()
        return username, password, database_host, database_name, database_port

# Get credentials from secret file
username, password, database_host, database_name, database_port = get_credentials()

# Database connection URL
DATABASE_URL = f"postgresql://{username}:{password}@{database_host}:{database_port}/{database_name}"

# Create engine
engine = create_engine(DATABASE_URL)

## Retrieve the data

In [59]:
# function to mask the encounter numbers
import hashlib
import secrets

def hash_csn(df):
    """
    Consistently hash CSN values in a dataframe
    Returns a new dataframe with hashed CSN column
    """
    # Create a copy to avoid modifying original
    df_hashed = df.copy()
    
    # Use a fixed salt for consistency
    FIXED_SALT = "your_fixed_salt_here"  # You can change this value
    
    def hash_value(value):
        if pd.isna(value):
            return None
        salted = f"{str(value)}{FIXED_SALT}".encode()
        return hashlib.sha256(salted).hexdigest()[:12]
    
    # Apply the hash function to the CSN column
    df_hashed['csn'] = df_hashed['csn'].apply(hash_value)
    
    return df_hashed


In [95]:
import random
from datetime import timedelta

with open('../../seed', 'r') as file:
    seed = file.readline().strip()

def shift_dates_inplace(df, seed, min_weeks=52, max_weeks=52*2):
    df_copy = df.copy()

    """Shift datetime columns in place"""
    random.seed(seed)
    weeks_to_add = random.randint(min_weeks, max_weeks)
    shift_delta = timedelta(weeks=weeks_to_add)

    datetime_cols = df_copy.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns

    for col in datetime_cols:
        df_copy[col] = df_copy[col].apply(lambda x: x + shift_delta if pd.notna(x) else x)

    return df_copy

In [92]:
# get the data
from pathlib import Path
import pandas as pd
import hashlib

# set date range
arrived_after = '2024-01-01'
arrived_before = '2024-01-31'

# create parameters dictionary
params = {
    'arrived_after': arrived_after,
    'arrived_before': arrived_before
}

# set up SQL query
SQL_DIR = Path("/home/jovyan/work/zella/zbeds/sql")
subquery = (SQL_DIR / "EMAP_ed_subquery.sql").read_text()
mainquery = (SQL_DIR / "EMAP_test_script.sql").read_text()
final_query = mainquery.replace('[subquery]', f'({subquery})')

# execute the combined query
df = pd.read_sql(
    final_query,
    engine,
    params=params
)

# Hash the csns before display
df = hash_csn(df)

# shift the dates before display
df = shift_dates_inplace(df, seed)

## Identify moment of departure from ED

In [101]:
df.sort_values(['csn', 'location_arrival'], inplace= True)

In [108]:
# create a mask for ED locations using string operations
ed_mask = (
    df['location_string'].str.startswith('ED^') |
    df['location_string'].str.startswith('1020100166^') |
    df['location_string'].str.startswith('1020100170^')
)

# Filter for ED locations and group by CSN to find first departure
first_ed_departure = (
    df[ed_mask]
    .groupby('csn')['location_departure']
    .max()
    .reset_index()
    .rename(columns={'location_departure': 'first_ed_departure'})
)

# merge this back with original dataframe:
df_with_departure = df.merge(
    first_ed_departure,
    on='csn',
    how='left'
)

## Identify whether patient was admitted

In [152]:
# Calculate admission status
def determine_admission(group):
    # Get rows after first ED departure
    post_ed = group[group['location_arrival'] >= group['first_ed_departure']]
    return len(post_ed) > 0

# Group by CSN and apply the admission check
admissions = (
    df_with_departure
    .groupby('csn')
    .apply(determine_admission)
    .reset_index()
    .rename(columns={0: 'is_admitted'})
)

# Merge admission status back to dataframe
df_with_admission_status = df_with_departure.merge(
    admissions,
    on='csn',
    how='left'
)

In [162]:
df_with_admission_status.head(20)

,csn,patient_class,presentation_datetime,hospital_arrival,hospital_departure,arrival_method,location_arrival,location_departure,location_string,first_ed_departure,is_admitted
0,0003871caf20,INPATIENT,2025-03-05 18:30:17,2025-03-05 19:16:00,2025-03-06 19:45:00,Walk-in,2025-03-05 18:30:48,2025-03-05 19:16:00,ED^null^null,2025-03-06 01:27:00,True
1,0003871caf20,INPATIENT,2025-03-05 18:30:17,2025-03-05 19:16:00,2025-03-06 19:45:00,Walk-in,2025-03-05 19:16:00,2025-03-05 21:09:00,ED^UCHED UTC POOL01^UTC WR,2025-03-06 01:27:00,True
2,0003871caf20,INPATIENT,2025-03-05 18:30:17,2025-03-05 19:16:00,2025-03-06 19:45:00,Walk-in,2025-03-05 21:09:00,2025-03-06 01:27:00,1020100166^SDEC BY01^CH07,2025-03-06 01:27:00,True
3,0003871caf20,INPATIENT,2025-03-05 18:30:17,2025-03-05 19:16:00,2025-03-06 19:45:00,Walk-in,2025-03-06 01:27:00,2025-03-06 19:45:00,1020100165^T14SASU SR29^SR29-29,2025-03-06 01:27:00,True
4,00038d22d7a9,EMERGENCY,2025-02-20 12:23:18,2025-02-20 12:40:00,2025-02-20 13:46:00,Walk-in,2025-02-20 12:23:26,2025-02-20 12:40:00,ED^null^null,2025-02-20 13:46:00,False
5,00038d22d7a9,EMERGENCY,2025-02-20 12:23:18,2025-02-20 12:40:00,2025-02-20 13:46:00,Walk-in,2025-02-20 12:40:00,2025-02-20 13:46:00,ED^UCHED UTC POOL02^UTC TZ,2025-02-20 13:46:00,False
6,000639d6912b,INPATIENT,2025-03-10 13:44:35,2025-03-10 14:02:00,2025-03-14 18:00:00,Public Trans,2025-03-10 13:44:42,2025-03-10 13:44:44,ED^null^null,2025-03-10 20:27:00,True
7,000639d6912b,INPATIENT,2025-03-10 13:44:35,2025-03-10 14:02:00,2025-03-14 18:00:00,Public Trans,2025-03-10 13:44:44,2025-03-10 14:02:00,ED^null^null,2025-03-10 20:27:00,True
8,000639d6912b,INPATIENT,2025-03-10 13:44:35,2025-03-10 14:02:00,2025-03-14 18:00:00,Public Trans,2025-03-10 14:02:00,2025-03-10 14:21:00,ED^UCHED RAT CHAIR^RAT-CHAIR,2025-03-10 20:27:00,True
9,000639d6912b,INPATIENT,2025-03-10 13:44:35,2025-03-10 14:02:00,2025-03-14 18:00:00,Public Trans,2025-03-10 14:21:00,2025-03-10 14:34:00,ED^UCHED RAT03^RAT-03,2025-03-10 20:27:00,True


In [154]:
# Modify final_df to include only ED rows

df_final = df_with_admission_status[df_with_admission_status.location_departure <= df_with_admission_status.first_ed_departure]

## Set up an array of snapshot datetimes

In [118]:
# indicate whether the notebook is being run locally for UCLH or with public datasets
uclh = False
from patientflow.load import set_file_paths
from patientflow.load import load_config_file

# set file locations
data_folder_name = 'data-uclh' if uclh else 'data-public'
data_file_path, media_file_path, model_file_path, config_path = set_file_paths(
        train_dttm = None, data_folder_name = data_folder_name, uclh = uclh, from_notebook=True, inference_time = False)

# load params
params = load_config_file(config_path)

snapshot_times = params["prediction_times"]



Configuration will be loaded from: /home/jovyan/work/zella/patientflow/config.yaml
Data files will be loaded from: /home/jovyan/work/zella/patientflow/data-public
Trained models will be saved to: /home/jovyan/work/zella/patientflow/trained-models
Images will be saved to: /home/jovyan/work/zella/patientflow/notebooks/img


In [137]:
from datetime import datetime, time, timedelta
import random
import pandas as pd

def get_shifted_snapshot_dates(arrived_after, arrived_before, seed, min_weeks=52, max_weeks=52*2):
    # First get the original dates
    original_dates = pd.date_range(
        start=arrived_after, 
        end=arrived_before, 
        freq="D"
    ).date.tolist()[:-1]
    
    # Apply the same shift
    random.seed(seed)
    weeks_to_add = random.randint(min_weeks, max_weeks)
    shift_delta = timedelta(weeks=weeks_to_add)
    
    # Shift each date
    shifted_dates = [date + shift_delta for date in original_dates]
    
    return shifted_dates

snapshot_dates = get_shifted_snapshot_dates(arrived_after, arrived_before, seed)


## Create snapshots dataset


In [166]:
from datetime import datetime, time
import pandas as pd

def create_snapshots(df, snapshot_times, snapshot_dates):
    # Create empty list to store all results
    all_results = []
    
    # For each combination of date and time
    for date in snapshot_dates:
        for hour, minute in snapshot_times:
            snapshot_datetime = datetime.combine(
                date, 
                time(hour=hour, minute=minute)
            )
            
            # Filter dataframe for this snapshot
            mask = (df['location_arrival'] <= snapshot_datetime) & (df['location_departure'] > snapshot_datetime)
            snapshot_df = df[mask].copy()  # Create copy to avoid SettingWithCopyWarning
            
            # Add snapshot information columns
            snapshot_df['snapshot_date'] = date
            snapshot_df['snapshot_time'] = [(hour, minute)] * len(snapshot_df)
            
            # Append to results list
            all_results.append(snapshot_df)
    
    # Combine all results into single dataframe
    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)
        snapshot_cols = ['snapshot_date', 'snapshot_time']
        other_cols = [col for col in final_df.columns if col not in snapshot_cols]
        final_df = final_df[snapshot_cols + other_cols]
    else:
        # Create empty dataframe with correct columns if no results found
        final_df = pd.DataFrame(columns=list(df.columns) + ['snapshot_date', 'snapshot_time', 'snapshot_datetime'])
    
    return final_df.drop(columns = ['patient_class', 'presentation_datetime', 'hospital_arrival', 'hospital_departure', 'location_arrival', 'location_departure', 'first_ed_departure'])

snapshots_df = create_snapshots(df_final, snapshot_times, snapshot_dates)

In [167]:
# doesn't appear in snapshots because this patient was whizzed to the stroke unit
snapshots_df[snapshots_df.csn=='000639d6912b']

,snapshot_date,snapshot_time,csn,arrival_method,location_string,is_admitted
7879,2025-03-10,"(15, 30)",000639d6912b,Public Trans,ED^UCHED RESUS04^04-RESUS,True


In [168]:
snapshots_df#.shape

,snapshot_date,snapshot_time,csn,arrival_method,location_string,is_admitted
0,2025-02-17,"(6, 0)",04ecf3858397,Walk-in,ED^UCHED UTC POOL01^UTC WR,False
1,2025-02-17,"(6, 0)",08e9755fa1de,Ambulance,ED^UCHED MAJCH01^MAJ-CHAIR,False
2,2025-02-17,"(6, 0)",0eab59060fc4,Public Trans,ED^UCHED UTC POOL02^UTC TZ,False
3,2025-02-17,"(6, 0)",1690e1749b1e,Public Trans,ED^UCHED MAJCH01^MAJ-CHAIR,False
4,2025-02-17,"(6, 0)",1b4502224fa2,Public Trans,ED^UCHED UTC POOL02^UTC TZ,False
...,...,...,...,...,...,...
12329,2025-03-18,"(22, 0)",f6473405710a,Walk-in,ED^UCHED RAT06^RAT-06,True
12330,2025-03-18,"(22, 0)",f88d666c7da7,Walk-in,ED^UCHED MAJ WR POOL^MAJ-WR,False
12331,2025-03-18,"(22, 0)",fafaf41b2b40,Ambulance,ED^NON COVID MAJORS 22^22-NON COVID MAJORS,False
12332,2025-03-18,"(22, 0)",fefa0f297f12,Walk-in,ED^UCHED MAJCH01^MAJ-CHAIR,False
